# 01_info.ipynb — UC-1 정보 확인·저장

`info()` 함수와 T-1 ~ T-20 검증.


In [ ]:
import os
import json
import youtube_kit as yk

from pathlib import Path
# 루트의 outputs 에 저장한다. notebooks 에서 열어도, 루트에서 실행해도 같은 곳이다
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out_dir = str(ROOT / "outputs")
os.makedirs(out_dir, exist_ok=True)
print("youtube_kit version:", yk.__version__)


## T-1 (bWPXADZylm0): 기본 경로, 챕터 20개, 포맷 및 자막 목록


In [2]:
info_t1 = yk.info("bWPXADZylm0", out_dir=out_dir)
meta_path_t1 = os.path.join(out_dir, "bWPXADZylm0.meta.json")
assert os.path.exists(meta_path_t1)
assert len(info_t1["chapters"]) == 20
assert any(f["id"] == "137" for f in info_t1["video_formats"])
assert any(f["id"] == "140-1" and f["original"] is True for f in info_t1["audio_formats"])
assert any(f["id"] == "140-0" and f["original"] is False for f in info_t1["audio_formats"])
assert any(s["id"] == "manual:ko" for s in info_t1["subtitles"])
assert any(s["id"] == "auto:ko-orig" for s in info_t1["subtitles"])
print("T-1 verified: chapters=20, formats and subtitles present.")


T-1 verified: chapters=20, formats and subtitles present.


## T-2 (xyTPUdJhxLM): 긴 수동 자막 키


In [3]:
info_t2 = yk.info("xyTPUdJhxLM", out_dir=out_dir)
assert any(s["id"] == "manual:ko-1tAUxhK-Qig" for s in info_t2["subtitles"])
print("T-2 verified: manual:ko-1tAUxhK-Qig present.")


T-2 verified: manual:ko-1tAUxhK-Qig present.


## T-3 (VbD8ITrJ6lg): 짧은 영상, 수동 자막 없음, 자동 자막 존재


In [4]:
info_t3 = yk.info("VbD8ITrJ6lg", out_dir=out_dir)
assert not any(s["kind"] == "manual" for s in info_t3["subtitles"])
assert any(s["id"] == "auto:ko-orig" for s in info_t3["subtitles"])
print("T-3 verified: no manual subtitles, auto:ko-orig present.")


T-3 verified: no manual subtitles, auto:ko-orig present.


## T-4 (DOTROnklS28): 3개 이상 음성 언어 트랙


In [5]:
info_t4 = yk.info("DOTROnklS28", out_dir=out_dir)
audio_langs = {f["language"] for f in info_t4["audio_formats"] if f["language"]}
assert len(audio_langs) >= 3
print("T-4 verified: audio languages:", audio_langs)


T-4 verified: audio languages: {'en-US', 'id', 'ko'}


## T-5 (XQk8jNM4zD8): 영어 원본 + 한국어 더빙, 4K


In [6]:
info_t5 = yk.info("XQk8jNM4zD8", out_dir=out_dir)
orig_track = next(f for f in info_t5["audio_formats"] if f["id"] == "140-1")
dub_track = next(f for f in info_t5["audio_formats"] if f["id"] == "140-0")
assert orig_track["original"] is True
assert dub_track["original"] is False
assert any(f["height"] == 2160 for f in info_t5["video_formats"])
print("T-5 verified: 140-1 original=True, 140-0 original=False, 4K video format exists.")


T-5 verified: 140-1 original=True, 140-0 original=False, 4K video format exists.


## T-6 (v629ci8lHnA): 많은 더빙 트랙


In [7]:
info_t6 = yk.info("v629ci8lHnA", out_dir=out_dir)
assert len(info_t6["audio_formats"]) > 15
print("T-6 verified: audio formats count:", len(info_t6["audio_formats"]))


T-6 verified: audio formats count: 100


## T-7 (lbzSN3fFV_Q): DRC 변형 및 여러 수동 자막


In [8]:
info_t7 = yk.info("lbzSN3fFV_Q", out_dir=out_dir)
assert any(f["drc"] is True for f in info_t7["audio_formats"])
manual_subs = [s for s in info_t7["subtitles"] if s["kind"] == "manual"]
assert len(manual_subs) >= 7
print("T-7 verified: DRC format present, manual subtitles count:", len(manual_subs))


T-7 verified: DRC format present, manual subtitles count: 7


## T-8 (KM4Xe6Dlp0Y): 수동 자막 48개, -sr 업스케일 포맷


In [9]:
info_t8 = yk.info("KM4Xe6Dlp0Y", out_dir=out_dir)
manual_subs_t8 = [s for s in info_t8["subtitles"] if s["kind"] == "manual"]
assert len(manual_subs_t8) == 48
assert any("-sr" in f["id"] for f in info_t8["video_formats"])
print("T-8 verified: manual subtitles count == 48, -sr format present.")


T-8 verified: manual subtitles count == 48, -sr format present.


## T-9 (SbvAaDN1bpE): 60fps 포맷, VP9, Opus


In [10]:
info_t9 = yk.info("SbvAaDN1bpE", out_dir=out_dir)
assert any(f.get("fps") == 60 for f in info_t9["video_formats"])
assert any("vp09" in f["vcodec"] or "vp9" in f["vcodec"] for f in info_t9["video_formats"])
assert any("opus" in f["acodec"] for f in info_t9["audio_formats"])
print("T-9 verified: 60fps, VP9, Opus formats present.")


T-9 verified: 60fps, VP9, Opus formats present.


## T-10 (gTPUhQG4hqs): 단일 음성 트랙 (original is None)


In [11]:
info_t10 = yk.info("gTPUhQG4hqs", out_dir=out_dir)
assert all(f["original"] is None for f in info_t10["audio_formats"])
print("T-10 verified: single audio track original is None.")


T-10 verified: single audio track original is None.


## T-11 (rlw8cH7ApT4): 비표준 해상도 (1042p)


In [12]:
info_t11 = yk.info("rlw8cH7ApT4", out_dir=out_dir)
assert any(f.get("height") == 1042 for f in info_t11["video_formats"])
print("T-11 verified: 1042p height format present.")


T-11 verified: 1042p height format present.


## T-12 (DDubtRhOEGw): 자막 전혀 없음, 빈 언어 값


In [13]:
info_t12 = yk.info("DDubtRhOEGw", out_dir=out_dir)
assert len(info_t12["subtitles"]) == 0
print("T-12 verified: 0 subtitles.")


T-12 verified: 0 subtitles.


## T-13 (JyEsEcsCqzM): Shorts 세로 영상 URL 정규화


In [14]:
info_t13 = yk.info("https://www.youtube.com/shorts/JyEsEcsCqzM", out_dir=out_dir)
assert info_t13["id"] == "JyEsEcsCqzM"
assert info_t13["url"] == "https://www.youtube.com/watch?v=JyEsEcsCqzM"
assert any(f.get("height") == 1920 for f in info_t13["video_formats"])
print("T-13 verified: shorts normalized to standard watch URL.")


T-13 verified: shorts normalized to standard watch URL.


## T-14 (ohYhCpOzLLE): 언어 None 빈 값, 자동 자막 1개


In [15]:
info_t14 = yk.info("ohYhCpOzLLE", out_dir=out_dir)
assert info_t14["language"] is None
auto_subs = [s for s in info_t14["subtitles"] if s["kind"] == "auto"]
assert len(auto_subs) == 1
print("T-14 verified: language is None, 1 auto subtitle.")


T-14 verified: language is None, 1 auto subtitle.


## T-15 (rFZHOHl-L8A): 라이브 방송 중인 영상 조회


In [16]:
info_t15 = yk.info("rFZHOHl-L8A", out_dir=out_dir)
assert info_t15["is_live"] is True
assert info_t15["duration_s"] is None
print("T-15 verified: is_live is True, duration_s is None.")


T-15 verified: is_live is True, duration_s is None.


## T-16 (Et9xVTOjgko): 비공개 영상 -> VideoUnavailable


In [17]:
try:
    yk.info("Et9xVTOjgko", out_dir=out_dir)
    assert False, "Expected VideoUnavailable"
except yk.VideoUnavailable as exc:
    print("T-16 verified:", type(exc).__name__, "raised.")


T-16 verified: VideoUnavailable raised.


ERROR: [youtube] Et9xVTOjgko: Private video


## T-17 (9_kPesKDlE4): 멤버십 전용 영상 -> VideoUnavailable


In [18]:
try:
    yk.info("9_kPesKDlE4", out_dir=out_dir)
    assert False, "Expected VideoUnavailable"
except yk.VideoUnavailable as exc:
    print("T-17 verified:", type(exc).__name__, "raised.")


T-17 verified: VideoUnavailable raised.


ERROR: [youtube] 9_kPesKDlE4: This video is available to this channel's members on level: 슈퍼 코더 (or any higher level). Join this channel to get access to members-only content and other exclusive perks.


## T-18 (재생목록 조회): playlist.json 생성 및 entries 확인


In [19]:
playlist_url = "https://www.youtube.com/playlist?list=PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm"
p18 = yk.info(playlist_url, out_dir=out_dir)
playlist_file = os.path.join(out_dir, "PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm.playlist.json")
assert os.path.exists(playlist_file)
assert len(p18["entries"]) == 103
assert all("id" in e and "url" in e and "title" in e for e in p18["entries"])
print("T-18 verified: playlist file created, entries count == 103.")


T-18 verified: playlist file created, entries count == 103.


## T-19 (존재하지 않는 11자리 ID): VideoUnavailable


In [20]:
try:
    yk.info("00000000000", out_dir=out_dir)
    assert False, "Expected VideoUnavailable"
except yk.VideoUnavailable as exc:
    print("T-19 verified:", type(exc).__name__, "raised.")


T-19 verified: VideoUnavailable raised.


ERROR: [youtube] 00000000000: This video is unavailable


## T-20 (잘못된 입력 형식): InvalidInput


In [21]:
try:
    yk.info("abc", out_dir=out_dir)
    assert False, "Expected InvalidInput for 'abc'"
except yk.InvalidInput as exc:
    print("T-20 verified for 'abc':", type(exc).__name__, "raised.")

try:
    yk.info("PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm", out_dir=out_dir)
    assert False, "Expected InvalidInput for bare playlist ID"
except yk.InvalidInput as exc:
    print("T-20 verified for bare playlist ID:", type(exc).__name__, "raised.")


T-20 verified for 'abc': InvalidInput raised.
T-20 verified for bare playlist ID: InvalidInput raised.


## T-6 · T-12 보강: 같은 언어라도 `140`과 `251`의 번호가 다르다, 음성 언어 값이 비어 있다

In [ ]:
ko140 = [f["id"] for f in info_t6["audio_formats"] if f["language"] == "ko" and f["id"].startswith("140-")]
ko251 = [f["id"] for f in info_t6["audio_formats"] if f["language"] == "ko" and f["id"].startswith("251-")]
assert ko140 and ko251
assert ko140[0].split("-")[1] != ko251[0].split("-")[1]
print("T-6 verified: ko audio", ko140[0], "vs", ko251[0])

assert all(f["language"] is None for f in info_t12["audio_formats"])
print("T-12 verified: audio language is None.")